In [39]:
%reset -f
import arcpy
arcpy.ResetEnvironments()
import os
from arcpy.sa import Slope

# check out Spatial Analyst license
arcpy.CheckOutExtension("Spatial")

'CheckedOut'

In [40]:
# ── PATHS ────────────────────────────────────────────────────────────────────────────────────────────

# Input rasters to calculate slope from
gbc_bathy = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_thesisextent.gdb\GBC15a_bathy_p26920_v6642"
usc_bathy = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\Jeremiah_NOAA_10meter_USCaribbean.gdb\USC10m_bathy_p26920_v6642"

# Output slope rasters
gbc_slope = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_thesisextent.gdb\GBC15a_slope_p26920_v6642"
usc_slope = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\Jeremiah_NOAA_10meter_USCaribbean.gdb\USC10m_slope_p26920_v6642"

In [41]:
# ── SETTINGS ─────────────────────────────────────────────────────────────────────────────────────────
arcpy.env.overwriteOutput = True    # overwrite existing outputs on rerun
arcpy.env.compression = "NONE"      # set compression to NONE for output rasters

In [42]:
# ── INSPECT INPUT BATHY RASTERS DATA ─────────────────────────────────────────────────────────────────

# Function to print all info (number of bands, pixel type, min, max, cell size, coordinate systems) for each raster
def print_raster_info(raster):

    # get number of bands info
    d = arcpy.Describe(raster)
    # get pixel type/min/max/cell size info (r = arcpy.Raster() loads raster dataset into a Raster object so arcpy can access stats)
    r = arcpy.Raster(raster)
    # get spatial reference info
    sr = d.spatialReference
    vcs = sr.VCS.name if sr.VCS else "NONE"

    # info print block for raster
    print(f"\n---------------- {os.path.basename(raster)} ----------------: \n")
    print(f"   Number bands = {d.bandCount}")
    print(f"   Pixel type = {r.pixelType}")
    print(f"   ⬇️ Min = {r.minimum}")
    print(f"   ⬆️ Max = {r.maximum}")
    print(f"   📐 Cell size = {r.meanCellWidth}")
    print(f"   Horizontal SR: {sr.name}")
    print(f"   Vertical SR: {vcs}")
    print(f"\n")


# Calculate Statistics (Data Management Tools): (default X/Y skip factor = 1; skip rasters with existing stats calculated)
for stats_path1 in [gbc_bathy, usc_bathy]:
    arcpy.management.CalculateStatistics(
        in_raster_dataset=stats_path1,
        skip_existing="SKIP_EXISTING"
    )

# Call function to print info for each input bathy raster
print_raster_info(gbc_bathy)
print_raster_info(usc_bathy)


---------------- GBC15a_bathy_p26920_v6642 ----------------: 

   Number bands = 1
   Pixel type = S16
   ⬇️ Min = -8619.0
   ⬆️ Max = 0.0
   📐 Cell size = 472.8452741843477
   Horizontal SR: NAD_1983_UTM_Zone_20N
   Vertical SR: VIVD09_height



---------------- USC10m_bathy_p26920_v6642 ----------------: 

   Number bands = 1
   Pixel type = F32
   ⬇️ Min = -6239.16943359375
   ⬆️ Max = 0.0
   📐 Cell size = 10.0
   Horizontal SR: NAD_1983_UTM_Zone_20N
   Vertical SR: VIVD09_height




In [43]:
# ── 1. CREATE SLOPE RASTERS FROM BATHY RASTERS ───────────────────────────────────────────────────────

# Slope (Spatial Analyst Tools): Function
def compute_slope(in_bathy, out_slope):
    # snap raster to input bathy raster grid so output aligns to same grid
    arcpy.env.snapRaster = in_bathy

    try:
        out_slope_ras = Slope(
            in_raster=in_bathy,
            # output slope values in degrees
            output_measurement="DEGREE",
            # "geodesic method produces a more accurate result, at the potential cost of an increase in processing time"
            method="GEODESIC",
        )
        # save output as new raster
        out_slope_ras.save(out_slope)
        print(f"✅ Slope raster successfully created: {out_slope} \n")

    # print warning if slope tool failed
    except Exception as e:
        print(f"❌[FAILURE] Slope raster creation failed for {os.path.basename(out_slope)}: {e} \n")


# Call function for GBC and USC rasters
compute_slope(gbc_bathy, gbc_slope)
compute_slope(usc_bathy, usc_slope)

# Clear settings in environment for subsequent blocks
arcpy.ClearEnvironment("snapRaster")

✅ Slope raster successfully created: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_thesisextent.gdb\GBC15a_slope_p26920_v6642 

✅ Slope raster successfully created: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\Jeremiah_NOAA_10meter_USCaribbean.gdb\USC10m_slope_p26920_v6642 



In [44]:
# ── INSPECT OUTPUT SLOPE RASTERS DATA ────────────────────────────────────────────────────────────────

# Calculate Statistics (Data Management Tools): (default X/Y skip factor = 1)
for stats_path2 in [gbc_slope, usc_slope]:
    arcpy.management.CalculateStatistics(in_raster_dataset=stats_path2)

# Call function to print info for each output slope raster
print_raster_info(gbc_slope)
print_raster_info(usc_slope)


---------------- GBC15a_slope_p26920_v6642 ----------------: 

   Number bands = 1
   Pixel type = F32
   ⬇️ Min = 1.0411026707402016e-08
   ⬆️ Max = 79.22057342529297
   📐 Cell size = 472.845274184348
   Horizontal SR: NAD_1983_UTM_Zone_20N
   Vertical SR: VIVD09_height



---------------- USC10m_slope_p26920_v6642 ----------------: 

   Number bands = 1
   Pixel type = F32
   ⬇️ Min = 8.323395792042554e-12
   ⬆️ Max = 89.75553131103516
   📐 Cell size = 10.0
   Horizontal SR: NAD_1983_UTM_Zone_20N
   Vertical SR: VIVD09_height


